In [ ]:
import numpy as np
import ot

mu = np.array([0.1, 0.2, 0.05, 0.15, 0.5])
nu = np.array([1.0])

C = np.array([[2],[1],[0.9],[1.2],[0.5]])

W1 = ot.emd2(mu,nu,C)

curv = 1 - W2/0.5
curv /= 0.5

print(W1, curv)

In [ ]:
import numpy as np
import ot

mu = np.array([0.1, 0.2, 0.05, 0.15, 0.5])
nu = np.array([0.05, 0.15, 0.2, 0.1, 0.5])

C = np.array([[2.5,2.5,2.2,2.2,2],[1.5,1.5,1.2,1.2,1],[1.4,1.4,1.1,1.1,0.9],[1.7,1.7,1.4,1.4,1.2],[1,1,0.7,0.7,0.5]])

W2 = ot.emd2(mu,nu,C)

curv = 1 - W2/0.5
curv /= 0.5

print(W2, curv)

In [ ]:
W2-W1

In [ ]:
import torch

# meta
num_q_heads = 4
num_kv_heads = 2
repeat = 2
head_dim = 2
seq = 3
B = 1

# Construct qh manually so it's easy to track
# shape: [B, q_heads, S, D]
qh = torch.tensor([[
    # q head 0
    [[1., 2.],
     [3., 4.],
     [5., 6.]],

    # q head 1
    [[2., 1.],
     [4., 3.],
     [6., 5.]],

    # q head 2
    [[1., 1.],
     [2., 2.],
     [3., 3.]],

    # q head 3
    [[2., 2.],
     [3., 3.],
     [4., 4.]],
]])

print("qh shape:", qh)

# ---- your pipeline ----
q_cost = qh.mean(dim=0)                # [q_heads, S, D]


# group into kv_heads
q_cost = q_cost.view(num_kv_heads, repeat, seq, head_dim)
print("\nafter view (kv_heads, repeat, S, D):", q_cost.shape)

# permute
q_cost = q_cost.permute(0, 3, 1, 2).contiguous()   # [kv_heads, D, repeat, S]
print("after permute:", q_cost.shape)

# flatten repeat*seq
q_cost = q_cost.view(num_kv_heads, head_dim, repeat * seq)
print("after final view:", q_cost.shape)

# ---- build output ----

out = torch.full(
    (num_kv_heads * head_dim, num_q_heads * seq),
    float("inf")
)

for kvh in range(num_kv_heads):
    r0 = kvh * head_dim
    r1 = (kvh + 1) * head_dim

    qh0 = kvh * repeat
    qh1 = (kvh + 1) * repeat

    c0 = qh0 * seq
    c1 = qh1 * seq

    out[r0:r1, c0:c1] = q_cost[kvh]

print("\nFinal out matrix:")
print(out)